# Agents Assignments - Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chebil/AI-course-book/blob/main/chapters/ch01_AgentAssignments_Solutions.ipynb)

⚠️ **INSTRUCTOR USE ONLY** - This notebook contains complete solutions to the assignments.

## Overview

This notebook contains fully implemented solutions for:

1. **Assignment 1**: Reflex Agent for a Traffic Light Controller
2. **Assignment 2**: Model-Based Agent for a Security Patrol

In [ ]:
# Setup - Run this cell first
import random
import time
from collections import deque

random.seed(42)
print("✓ Setup complete!")

---

## Assignment 1: Reflex Agent - Traffic Light Controller (40 points)

### Problem Description

Implement a **Simple Reflex Agent** that controls a traffic light at an intersection.

### Rules:
1. If current light is `'NS_GREEN'` and EW traffic is `'heavy'` while NS traffic is `'none'` or `'light'` → Switch to `'EW_GREEN'`
2. If current light is `'EW_GREEN'` and NS traffic is `'heavy'` while EW traffic is `'none'` or `'light'` → Switch to `'NS_GREEN'`
3. If both directions have `'heavy'` traffic → Keep current state
4. If both directions have `'none'` traffic → Keep current state
5. Otherwise, switch to the direction with heavier traffic

In [ ]:
# Traffic Light Environment (PROVIDED - Do not modify)

class TrafficIntersection:
    """Simulates a 4-way traffic intersection."""
    
    TRAFFIC_LEVELS = ['none', 'light', 'moderate', 'heavy']
    
    def __init__(self):
        self.light_state = 'NS_GREEN'  # Initial state
        self.ns_traffic = 'moderate'
        self.ew_traffic = 'moderate'
        self.time_in_state = 0
        self.total_wait_time = 0
        self.cars_passed = 0
    
    def get_percept(self):
        """Returns current traffic conditions."""
        return {
            'current_light': self.light_state,
            'ns_traffic': self.ns_traffic,
            'ew_traffic': self.ew_traffic
        }
    
    def apply_action(self, action):
        """Apply agent's action to change light state."""
        if action in ['NS_GREEN', 'EW_GREEN']:
            if action != self.light_state:
                self.light_state = action
                self.time_in_state = 0
            else:
                self.time_in_state += 1
        
        # Simulate traffic flow
        self._simulate_traffic_flow()
    
    def _simulate_traffic_flow(self):
        """Simulate cars passing and new cars arriving."""
        # Cars pass through green light
        if self.light_state == 'NS_GREEN':
            ns_idx = self.TRAFFIC_LEVELS.index(self.ns_traffic)
            self.cars_passed += ns_idx
            # Reduce NS traffic, increase EW traffic slightly
            if ns_idx > 0 and random.random() < 0.3:
                self.ns_traffic = self.TRAFFIC_LEVELS[ns_idx - 1]
            ew_idx = self.TRAFFIC_LEVELS.index(self.ew_traffic)
            if ew_idx < 3 and random.random() < 0.4:
                self.ew_traffic = self.TRAFFIC_LEVELS[ew_idx + 1]
                self.total_wait_time += ew_idx + 1
        else:
            ew_idx = self.TRAFFIC_LEVELS.index(self.ew_traffic)
            self.cars_passed += ew_idx
            if ew_idx > 0 and random.random() < 0.3:
                self.ew_traffic = self.TRAFFIC_LEVELS[ew_idx - 1]
            ns_idx = self.TRAFFIC_LEVELS.index(self.ns_traffic)
            if ns_idx < 3 and random.random() < 0.4:
                self.ns_traffic = self.TRAFFIC_LEVELS[ns_idx + 1]
                self.total_wait_time += ns_idx + 1
        
        # Random traffic changes
        if random.random() < 0.2:
            self.ns_traffic = random.choice(self.TRAFFIC_LEVELS)
        if random.random() < 0.2:
            self.ew_traffic = random.choice(self.TRAFFIC_LEVELS)
    
    def get_performance(self):
        """Returns performance metrics."""
        return {
            'cars_passed': self.cars_passed,
            'total_wait_time': self.total_wait_time,
            'efficiency': self.cars_passed / max(1, self.total_wait_time)
        }

print("✓ TrafficIntersection class loaded!")

In [ ]:
# ✅ SOLUTION: Traffic Light Reflex Agent

class TrafficLightReflexAgent:
    """
    A simple reflex agent for controlling traffic lights.
    
    The agent decides which direction gets the green light based
    solely on the current traffic conditions (no memory of past states).
    """
    
    TRAFFIC_PRIORITY = {'none': 0, 'light': 1, 'moderate': 2, 'heavy': 3}
    
    def __init__(self):
        self.name = "Traffic Light Reflex Agent"
    
    def get_traffic_level(self, traffic):
        """Convert traffic string to numeric level."""
        return self.TRAFFIC_PRIORITY.get(traffic, 0)
    
    def decide(self, percept):
        """
        Decide which light state to set based on current percept.
        
        Args:
            percept: dict with keys 'current_light', 'ns_traffic', 'ew_traffic'
        
        Returns:
            'NS_GREEN' or 'EW_GREEN'
        """
        current_light = percept['current_light']
        ns_traffic = percept['ns_traffic']
        ew_traffic = percept['ew_traffic']
        
        ns_level = self.get_traffic_level(ns_traffic)
        ew_level = self.get_traffic_level(ew_traffic)
        
        # Rule 1: If current is NS_GREEN and EW is heavy while NS is none/light -> EW_GREEN
        if current_light == 'NS_GREEN' and ew_level == 3 and ns_level <= 1:
            return 'EW_GREEN'
        
        # Rule 2: If current is EW_GREEN and NS is heavy while EW is none/light -> NS_GREEN
        if current_light == 'EW_GREEN' and ns_level == 3 and ew_level <= 1:
            return 'NS_GREEN'
        
        # Rule 3: If both heavy -> keep current
        if ns_level == 3 and ew_level == 3:
            return current_light
        
        # Rule 4: If both none -> keep current
        if ns_level == 0 and ew_level == 0:
            return current_light
        
        # Rule 5: Otherwise -> switch to heavier traffic direction
        if ew_level > ns_level:
            return 'EW_GREEN'
        elif ns_level > ew_level:
            return 'NS_GREEN'
        else:
            # Equal traffic levels, keep current
            return current_light

print("✓ TrafficLightReflexAgent SOLUTION loaded!")

In [ ]:
# Test the Traffic Light Reflex Agent

def test_traffic_agent():
    """Test the traffic light agent with various scenarios."""
    agent = TrafficLightReflexAgent()
    
    test_cases = [
        # (percept, expected_action, description)
        ({'current_light': 'NS_GREEN', 'ns_traffic': 'light', 'ew_traffic': 'heavy'}, 
         'EW_GREEN', "Heavy EW traffic, light NS -> should switch to EW"),
        
        ({'current_light': 'EW_GREEN', 'ns_traffic': 'heavy', 'ew_traffic': 'none'}, 
         'NS_GREEN', "Heavy NS traffic, no EW -> should switch to NS"),
        
        ({'current_light': 'NS_GREEN', 'ns_traffic': 'heavy', 'ew_traffic': 'heavy'}, 
         'NS_GREEN', "Both heavy -> should keep current (NS)"),
        
        ({'current_light': 'EW_GREEN', 'ns_traffic': 'none', 'ew_traffic': 'none'}, 
         'EW_GREEN', "Both none -> should keep current (EW)"),
        
        ({'current_light': 'NS_GREEN', 'ns_traffic': 'light', 'ew_traffic': 'moderate'}, 
         'EW_GREEN', "EW has more traffic -> should switch to EW"),
    ]
    
    passed = 0
    for percept, expected, description in test_cases:
        result = agent.decide(percept)
        status = "✓" if result == expected else "✗"
        if result == expected:
            passed += 1
        print(f"{status} {description}")
        print(f"   Expected: {expected}, Got: {result}")
    
    print(f"\n{'='*50}")
    print(f"Tests passed: {passed}/{len(test_cases)}")
    return passed == len(test_cases)

# Run tests
test_traffic_agent()

In [ ]:
# Run simulation with the agent

def run_traffic_simulation(agent, steps=50):
    """Run a full simulation of the traffic intersection."""
    env = TrafficIntersection()
    
    print(f"Running {steps}-step simulation with {agent.name}...\n")
    
    for step in range(steps):
        percept = env.get_percept()
        action = agent.decide(percept)
        env.apply_action(action)
        
        if step % 10 == 0:
            print(f"Step {step}: Light={percept['current_light']}, "
                  f"NS={percept['ns_traffic']}, EW={percept['ew_traffic']} "
                  f"-> Action: {action}")
    
    perf = env.get_performance()
    print(f"\n{'='*50}")
    print(f"Final Performance:")
    print(f"  Cars passed: {perf['cars_passed']}")
    print(f"  Total wait time: {perf['total_wait_time']}")
    print(f"  Efficiency score: {perf['efficiency']:.2f}")
    return perf

# Run simulation
agent = TrafficLightReflexAgent()
performance = run_traffic_simulation(agent)

---

## Assignment 2: Model-Based Agent - Security Patrol Agent (40 points)

### Problem Description

Implement a **Model-Based Reflex Agent** that patrols a building by:
- Tracking when each room was last visited
- Prioritizing unvisited rooms
- Responding to alerts

In [ ]:
# Building Environment (PROVIDED - Do not modify)

class BuildingEnvironment:
    """Simulates a building with 6 rooms for security patrol."""
    
    def __init__(self):
        self.num_rooms = 6
        self.agent_position = 0  # Start in room 0
        self.room_status = ['clear'] * self.num_rooms  # All rooms start clear
        self.steps = 0
        self.score = 0
        self.alerts_handled = 0
        self.rooms_checked = set()
    
    def get_percept(self):
        """Returns what the agent can see from current position."""
        # Agent can see current room and adjacent rooms
        visible_alerts = []
        for i in range(max(0, self.agent_position - 1), 
                       min(self.num_rooms, self.agent_position + 2)):
            if self.room_status[i] == 'alert':
                visible_alerts.append(i)
        
        return {
            'current_room': self.agent_position,
            'current_status': self.room_status[self.agent_position],
            'visible_alerts': visible_alerts,
            'num_rooms': self.num_rooms
        }
    
    def apply_action(self, action):
        """Move the agent and update environment."""
        self.steps += 1
        
        # Move agent
        if action == 'LEFT' and self.agent_position > 0:
            self.agent_position -= 1
        elif action == 'RIGHT' and self.agent_position < self.num_rooms - 1:
            self.agent_position += 1
        # STAY keeps position
        
        # Check current room (agent presence clears alerts)
        self.rooms_checked.add(self.agent_position)
        if self.room_status[self.agent_position] == 'alert':
            self.room_status[self.agent_position] = 'clear'
            self.alerts_handled += 1
            self.score += 20  # Bonus for handling alert
        else:
            self.score += 1  # Small score for patrolling
        
        # Random events: new alerts may appear
        if random.random() < 0.15:  # 15% chance of new alert
            alert_room = random.randint(0, self.num_rooms - 1)
            if alert_room != self.agent_position:
                self.room_status[alert_room] = 'alert'
    
    def get_performance(self):
        """Returns performance metrics."""
        coverage = len(self.rooms_checked) / self.num_rooms
        return {
            'score': self.score,
            'alerts_handled': self.alerts_handled,
            'coverage': coverage,
            'steps': self.steps
        }

print("✓ BuildingEnvironment class loaded!")

In [ ]:
# ✅ SOLUTION: Security Patrol Model-Based Agent

class SecurityPatrolAgent:
    """
    A model-based agent for security patrol.
    
    This agent maintains an internal model of:
    - Last visit time for each room
    - Current step count (to track time)
    """
    
    def __init__(self, num_rooms=6):
        self.name = "Security Patrol Model-Based Agent"
        self.num_rooms = num_rooms
        
        # Internal model: track when each room was last visited
        self.last_visit = {i: -100 for i in range(num_rooms)}
        self.current_step = 0
        
    def update_model(self, percept):
        """
        Update internal model based on new percept.
        """
        current_room = percept['current_room']
        
        # Record that we visited this room at this step
        self.last_visit[current_room] = self.current_step
        self.current_step += 1
    
    def get_least_recently_visited(self, current_room):
        """
        Find which adjacent room was visited least recently.
        
        Returns:
            'LEFT', 'RIGHT', or 'STAY'
        """
        left_room = current_room - 1
        right_room = current_room + 1
        
        # Check if we can go left or right
        can_go_left = left_room >= 0
        can_go_right = right_room < self.num_rooms
        
        if can_go_left and can_go_right:
            # Compare last visit times
            left_time = self.last_visit[left_room]
            right_time = self.last_visit[right_room]
            
            if left_time < right_time:
                return 'LEFT'  # Left was visited longer ago
            else:
                return 'RIGHT'  # Right was visited longer ago (or equal)
        elif can_go_left:
            return 'LEFT'
        elif can_go_right:
            return 'RIGHT'
        else:
            return 'STAY'
    
    def decide(self, percept):
        """
        Decide movement based on percept and internal model.
        
        Returns:
            'LEFT', 'RIGHT', or 'STAY'
        """
        # Step 1: Update internal model (record visit)
        self.update_model(percept)
        
        current_room = percept['current_room']
        visible_alerts = percept['visible_alerts']
        
        # Priority 1: If there's a visible alert, move toward it
        if visible_alerts:
            alert_room = visible_alerts[0]  # Handle first alert
            if alert_room < current_room:
                return 'LEFT'
            elif alert_room > current_room:
                return 'RIGHT'
            # Alert is in current room - will be handled automatically
        
        # Priority 2: Move toward least recently visited room
        return self.get_least_recently_visited(current_room)

print("✓ SecurityPatrolAgent SOLUTION loaded!")

In [ ]:
# Test the Security Patrol Agent

def test_patrol_agent():
    """Test the patrol agent with various scenarios."""
    agent = SecurityPatrolAgent(num_rooms=6)
    
    test_cases = [
        # (percept, expected_action, description)
        ({'current_room': 2, 'current_status': 'clear', 'visible_alerts': [3], 'num_rooms': 6},
         'RIGHT', "Alert to the right -> should move RIGHT"),
        
        ({'current_room': 3, 'current_status': 'clear', 'visible_alerts': [2], 'num_rooms': 6},
         'LEFT', "Alert to the left -> should move LEFT"),
        
        ({'current_room': 0, 'current_status': 'clear', 'visible_alerts': [], 'num_rooms': 6},
         'RIGHT', "At left edge, no alerts -> should move RIGHT"),
        
        ({'current_room': 5, 'current_status': 'clear', 'visible_alerts': [], 'num_rooms': 6},
         'LEFT', "At right edge, no alerts -> should move LEFT"),
    ]
    
    passed = 0
    for percept, expected, description in test_cases:
        result = agent.decide(percept)
        is_correct = result == expected
        status = "✓" if is_correct else "✗"
        if is_correct:
            passed += 1
        print(f"{status} {description}")
        print(f"   Expected: {expected}, Got: {result}")
    
    print(f"\n{'='*50}")
    print(f"Tests passed: {passed}/{len(test_cases)}")
    return passed >= 3

# Run tests
test_patrol_agent()

In [ ]:
# Run patrol simulation with the agent

def run_patrol_simulation(agent, steps=30):
    """Run a patrol simulation."""
    env = BuildingEnvironment()
    
    print(f"Running {steps}-step simulation with {agent.name}...\n")
    print(f"{'Step':>4} | {'Room':>4} | {'Status':>8} | {'Alerts':>10} | {'Action':>6}")
    print("-" * 50)
    
    for step in range(steps):
        percept = env.get_percept()
        action = agent.decide(percept)
        
        alerts_str = str(percept['visible_alerts']) if percept['visible_alerts'] else "None"
        print(f"{step:>4} | {percept['current_room']:>4} | {percept['current_status']:>8} | "
              f"{alerts_str:>10} | {action:>6}")
        
        env.apply_action(action)
    
    perf = env.get_performance()
    print(f"\n{'='*50}")
    print(f"Final Performance:")
    print(f"  Total score: {perf['score']}")
    print(f"  Alerts handled: {perf['alerts_handled']}")
    print(f"  Room coverage: {perf['coverage']*100:.0f}%")
    return perf

# Run simulation
agent = SecurityPatrolAgent()
performance = run_patrol_simulation(agent)

---

## Reflection Questions - Sample Answers (20 points)

### Question 1 (5 points)

**What are the key limitations of the reflex agent (Traffic Light Controller) compared to the model-based agent (Security Patrol)? Give specific examples from your implementations.**

**Sample Answer:**

The reflex agent has several key limitations:

1. **No memory**: The traffic light agent makes decisions based solely on current percepts. It cannot remember that it just switched the light, which could lead to rapid oscillation between states if traffic conditions fluctuate quickly.

2. **No history tracking**: In the traffic example, the agent cannot remember past traffic patterns to identify trends.

3. **Reactive only**: The reflex agent can only react to current conditions, not anticipate future needs.

In contrast, the patrol model-based agent:
- **Remembers visit history**: Tracks when each room was last visited using `last_visit` dictionary
- **Makes informed decisions**: Chooses to patrol rooms that haven't been checked recently
- **Avoids redundancy**: Won't keep patrolling the same room repeatedly

Example: If the patrol agent is in room 3, it checks `last_visit[2]` vs `last_visit[4]` to decide which adjacent room needs attention. A reflex agent would have no way to know which room was visited more recently.

### Question 2 (5 points)

**How does maintaining an internal model (visit history) help the patrol agent make better decisions? What would happen if it only used current percepts like a reflex agent?**

**Sample Answer:**

The internal model provides several advantages:

1. **Efficient coverage**: By tracking when each room was last visited, the agent ensures all rooms get regular attention. It prioritizes rooms that haven't been checked in a while.

2. **Avoids redundant patrolling**: Without memory, a reflex agent might oscillate between two rooms while ignoring others entirely.

3. **Better resource allocation**: The agent can make smarter decisions about where to go next based on patrol history.

**Without the internal model**, the agent would:
- Have no way to know which rooms need attention
- Possibly get stuck patrolling the same few rooms
- Miss covering the entire building effectively
- Have to rely on simple rules like "always go right" which would be inefficient

**Example**: Consider a 6-room building. A reflex agent at room 2 with no alerts visible has no good basis for choosing LEFT or RIGHT. The model-based agent checks: "Room 1 was visited 5 steps ago, Room 3 was visited 2 steps ago" and correctly chooses LEFT to check the neglected room.

### Question 3 (5 points)

**Could the traffic light controller benefit from being a model-based agent? What kind of internal model would you add, and how would it improve performance?**

**Sample Answer:**

Yes, the traffic light controller would significantly benefit from being model-based. Here's what could be added:

**Internal Model Components:**

1. **Traffic history**: Track traffic patterns over time to detect:
   - Rush hour patterns (high NS traffic 7-9 AM and 5-7 PM)
   - Weekend vs. weekday differences
   - Seasonal variations

2. **Time-based predictions**: Knowing it's 4:55 PM, predict heavy traffic soon

3. **Queue length model**: Estimate how many cars are waiting, not just traffic density

4. **Light change history**: Track how long since last change to prevent rapid switching

**Performance Improvements:**

- **Proactive switching**: Change to NS_GREEN before rush hour starts
- **Adaptive timing**: Longer green phases during predicted heavy traffic
- **Fairness**: Ensure no direction waits too long by tracking wait times
- **Reduced oscillation**: Minimum time between switches prevents confusion
- **Emergency response**: Could learn to detect ambulance patterns

### Question 4 (5 points)

**For each agent, identify the PEAS components (Performance measure, Environment, Actuators, Sensors):**

**Sample Answer:**

**Traffic Light Controller:**
- **Performance**: 
  - Maximize cars passed through intersection
  - Minimize total wait time across all directions
  - Efficiency ratio (cars passed / wait time)
- **Environment**: 
  - 4-way intersection
  - Dynamic traffic flow (stochastic arrivals)
  - Discrete state space
- **Actuators**: 
  - Traffic light control (switch between NS_GREEN and EW_GREEN)
- **Sensors**: 
  - Traffic density sensors for each direction (none/light/moderate/heavy)
  - Current light state

**Security Patrol Agent:**
- **Performance**: 
  - Maximize total score
  - Maximize alerts handled
  - Maximize room coverage (visit all rooms)
- **Environment**: 
  - Building with 6 rooms in a line
  - Random alert events
  - Discrete state space
- **Actuators**: 
  - Movement control (LEFT, RIGHT, STAY)
- **Sensors**: 
  - Current room position
  - Current room status (clear/alert)
  - Visible alerts in adjacent rooms

---

## ✅ All Solutions Complete

This notebook contains:
- ✅ Full implementation of `TrafficLightReflexAgent.decide()`
- ✅ All traffic agent test cases passing
- ✅ Full implementation of `SecurityPatrolAgent` with all methods:
  - `update_model()` - Records room visit times
  - `get_least_recently_visited()` - Finds neglected rooms

  - `decide()` - Handles alerts and patrols efficiently- ✅ Complete answers to all 4 reflection questions
- ✅ All patrol test cases passing